# MemeX on Colab GPU

Runtime -> Change runtime type -> **T4 GPU**, then Run all.
Cell 1 asks for `memex_code.zip` (the project folder, no models/venv).
Results land in `/content/out` and are zipped for download at the end.

In [ ]:
import torch, os, zipfile
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to T4 GPU')
from google.colab import files
up = files.upload()            # pick memex_code.zip
name = list(up)[0]
with zipfile.ZipFile(name) as z:
    z.extractall('/content/MemeX')
os.chdir('/content/MemeX')
os.makedirs('/content/out', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
print(sorted(os.listdir('.')))

In [ ]:
!pip install -q -U transformers huggingface_hub
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3-0.6B', local_dir='/content/Qwen3-0.6B')

import urllib.request
fic = '/content/data/calibration.txt'
text = ''
for u in ['https://www.gutenberg.org/files/2600/2600-0.txt',
          'https://www.gutenberg.org/files/1342/1342-0.txt',
          'https://www.gutenberg.org/files/84/84-0.txt']:
    try:
        text += urllib.request.urlopen(u, timeout=90).read().decode('utf-8', 'ignore')
    except Exception as e:
        print('skip', u, e)
open(fic, 'w', encoding='utf-8').write(text)
print('fiction chars:', len(text))
!python memex/make_corpus.py --fiction /content/data/calibration.txt \
  --out /content/data/corpus_mixed.txt --synth-chars 4000000 --fiction-chars 4000000

In [ ]:
# stage B: spectral transplant basis (1M tokens, minutes on T4)
!python memex/transplant.py --model /content/Qwen3-0.6B \
  --text /content/data/corpus_mixed.txt --out /content/out \
  --max-tokens 1000000 --chunk-len 4096 --device cuda

In [ ]:
# future-attention labels + salience head
!python memex/future_attention.py --model /content/Qwen3-0.6B \
  --text /content/data/corpus_mixed.txt --out /content/out/future_attn \
  --max-tokens 400000 --device cuda
!python memex/salience_probe.py --data /content/out/future_attn --holdout 4

In [ ]:
# needle benchmark at real scale: long haystack, many cases, all policies
!python bench/needle.py --model /content/Qwen3-0.6B \
  --results /content/out --futattn /content/out/future_attn \
  --haystack 16000 --window 1024 --sinks 4 --notebook 128 \
  --ranks 64 128 --depths 0.1 0.4 0.7 0.9 --per-depth 4 \
  --distractors 16 --probe-len 128 --topk-probe 64 --device cuda

In [ ]:
import json, collections, shutil
tally = collections.defaultdict(list)
for line in open('/content/out/needle_log.jsonl'):
    r = json.loads(line)
    tally[r['cond']].append(r['ok'])
for cond, v in sorted(tally.items()):
    print(f'{cond:>18}: {sum(v)}/{len(v)}  ({sum(v)/len(v):.2f})')
shutil.make_archive('/content/memex_results', 'zip', '/content/out')
from google.colab import files
files.download('/content/memex_results.zip')